In [ ]:

# 获取所有主题的字典 {topic_id: [(word, score), ...]}
all_topics_dict = topic_model.get_topics()
# 过滤掉 -1 (噪声)，并将每个词列表转为逗号连接的字符串
clean_topics_dict = {
    topic_id: ",".join([word for word, score in words_list])
    for topic_id, words_list in all_topics_dict.items()
    if topic_id != -1
}

for item in clean_topics_dict.items():
    print(item)
    
# 检查结果
print(f'已获取{len(clean_topics_dict)}个主题的详细信息，现在通过AI判断主题名称……') 

In [ ]:
# AI识别
system_prompt = """
你是一名专业的国防专利与技术分析专家，擅长从技术关键词中提炼核心主题方向。

【输入数据格式】
用户将提供一个Python字典：
- 键（key）：主题索引编号（整数，如0, 1, 2...）
- 值（value）：该主题的10个英文关键词字符串，以英文逗号分隔
- 关键词已按重要性降序排列

【核心任务】
为每个主题生成一个中文主题名称，要求：
1. **保守概括**：基于关键词的共同技术内涵，提取"上位技术方向"
2. **权重优先**：主要依据前3-5个高权重关键词，后部关键词仅参考
3. **避免扩展**：不引入新概念，不扩展应用场景，不做过度推断
4. **处理模糊**：若关键词分散，使用更抽象的名称（如"综合技术"）

【主题命名规则】
- 名称应体现国防/军事技术特点
- 使用标准技术术语，避免口语化
- 长度控制在6-15个汉字为宜
- 格式："{主题序号}. {名称}"，序号从1开始连续编号

【输出要求】
- 仅输出合法的JSON对象，无任何额外文本
- JSON结构：{"主题索引": "序号.主题名称"}
- 主题索引与输入保持一致
- 示例输出：{"0": "1.雷达探测", "1": "2.复合材料制备","2": "3.音频处理与降噪"}

【注意事项】
- 关键词可能存在词形变化（单复数等），理解其核心语义
- 国防领域特有术语应保留专业性
"""
user_prompt = f'''
请分析以下主題关键词字典，
为每个主题生成对应的中文主题名称，
并返回 JSON 字典：\n{clean_topics_dict}
'''

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import re
# 加载 .env 文件
load_dotenv(".env")
API_KEY=os.environ.get('DEEPSEEK_API_KEY')
deepseek_chat_model = "deepseek-chat" # DeepSeek-V3.2的非思考模式
client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.deepseek.com")
# AI识别智能体参数设置：创建专门用于地址推理的 LLM 实例
Deepseek_reasponse = client.chat.completions.create(
    model=deepseek_chat_model,
    messages=[ # 对话消息列表
        {"role": "system", "content": system_prompt}, # 系统提示词，定义助手的行为
        {"role": "user", "content": user_prompt},
    ],
    response_format={'type': 'json_object'}, #强制json格式返回
    stream=False # 非流式响应（一次性返回完整结果）
)
content = json.loads(Deepseek_reasponse.choices[0].message.content)
#content = re.sub(r"^```json\s*|\s*```$", "", content.strip())
# 将键转为整型
formatted_labels = {int(k): v for k, v in content.items()}
print(formatted_labels)
# 使用自定义标签（需要先设置）
topic_model.set_topic_labels(formatted_labels)